<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 06 - Extração de Informação (Parte 1)</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2>1. Reconhecimento de Entidades Nomeadas</h2>
<p>O Reconhecimento de Entidades Nomeadas (NER, do inglês Named Entity Recognition) consiste na identificação de nomes próprios em texto e na classificação em categorias, geralmente nomes de pessoas, organizações ou locais (cidades, estados, países, etc). Trata-se de uma tarefa essencial para extração de informação automática, uma vez que, com isso, é possível compreender quais entidades estão sendo tratadas no texto, auxiliando na compreensão de seu assunto como um todo.</p>

<p>O principal desafio está em resolver certas ambiguidades, além de considerar as palavras próximas para escolher a qual categoria uma determinada entidade pertence. Considere o exemplo abaixo:</p>

"Dr. Rodolfo S. Silva, um renomado cardiologista, adquiriu um novo consultório na Av. Edson A. Nascimento pela quantia de R$ 4.5 milhões, que atenderá pacientes do S.U.S. e do plano Saúde Mais Inc."

Nessa frase, temos as seguintes entidades:
* Rodolfo S. Silva (PESSOA)
* Av. Edson A. Nascimento (LOCALIDADE)
* S.U.S. (ORGANIZAÇÃO)
* Saúde Mais Inc. (ORGANIZAÇÃO)

Perceba que NER traz os seguintes desafios:
* Identificar qual palavra faz parte de uma entidade;
* Identificar onde começa uma entidade;
* Identificar onde termina uma entidade;
* Classificar uma entidade.

Há diversas bibliotecas para extração de entidades nomeadas disponíveis porém, a identificação de entidades em um corpus de domínio específico ou a identificação de entidades de categorias diferentes das padrão pode requerer o treinamento de um modelo de NER. Imagine, por exemplo, que se deseja extrair nomes de advogados e seus clientes em uma base de documentos jurídicos. Perceba que todos são nomes próprios, mas as palavras que os circundam devem ser diferentes, de uma forma geral. Nesse caso, é necessário treinar um modelo para isso, uma vez que os modelos padrão não atendem esse tipo de categoria.

Nesta aula veremos como fazer isso utilizando o algoritmo CRF (Conditional Random Field), uma das abordagens mais comuns para esse problema.

<h2>2. Algoritmo CRF (Conditional Random Field)</h2>

<p>Trata-se de um modelo discriminativo e probabilístico que trabalha com sequências. Em outras palavras, as amostras anterior e seguinte são levadas em consideração. A expressão abaixo ilustra esse modelo:</p>


![](https://cdn-images-1.medium.com/max/800/1*srzbO2c_h5FOZ3Kk0nhjpQ.gif)
<p fontsize='small'>Fonte: https://medium.com/ml2vec/overview-of-conditional-random-fields-68a2a20fa541</p>

No caso, <i>f</i> é o conjunto de atributos, que deve ser codificado e lambda é o conjunto de pesos aprendido pelo algoritmo. Exemplos de atributos para NER:

f1(s,i,li,li−1)=1 Se a primeira letra de li é maiúscula; senão 0

f2(s,i,li,li−1)=1 Se li é composta somente por letras maíusculas; senão 0

f3(s,i,li,li−1)=1 Se a POS Tag de li for NOUN; senão 0




Vamos estudar um exemplo de um modelo de NER em Espanhol para entidades do corpus CoNLL2002 (um corpus com entidades anotadas, disponibilizado em desafio apresentado na 6th Conference on Natural Language Learning, 2002), inspirado no código disponível em https://eli5.readthedocs.io/en/latest/tutorials/sklearn_crfsuite.html

In [ ]:
!pip install sklearn_crfsuite

In [ ]:
!pip install eli5

In [1]:
import nltk
import sklearn_crfsuite
import eli5

In [2]:
train_sents = list(nltk.corpus.conll2002.iob_sents('esp.train'))
test_sents = list(nltk.corpus.conll2002.iob_sents('esp.testb'))

In [3]:
import pprint

pprint.pprint(len(train_sents))
pprint.pprint(len(test_sents))

pprint.pprint(train_sents[2])
pprint.pprint(test_sents[2])

8323
1517
[('El', 'DA', 'O'),
 ('Abogado', 'NC', 'B-PER'),
 ('General', 'AQ', 'I-PER'),
 ('del', 'SP', 'I-PER'),
 ('Estado', 'NC', 'I-PER'),
 (',', 'Fc', 'O'),
 ('Daryl', 'VMI', 'B-PER'),
 ('Williams', 'NC', 'I-PER'),
 (',', 'Fc', 'O'),
 ('subrayó', 'VMI', 'O'),
 ('hoy', 'RG', 'O'),
 ('la', 'DA', 'O'),
 ('necesidad', 'NC', 'O'),
 ('de', 'SP', 'O'),
 ('tomar', 'VMN', 'O'),
 ('medidas', 'NC', 'O'),
 ('para', 'SP', 'O'),
 ('proteger', 'VMN', 'O'),
 ('al', 'SP', 'O'),
 ('sistema', 'NC', 'O'),
 ('judicial', 'AQ', 'O'),
 ('australiano', 'AQ', 'O'),
 ('frente', 'RG', 'O'),
 ('a', 'SP', 'O'),
 ('una', 'DI', 'O'),
 ('página', 'NC', 'O'),
 ('de', 'SP', 'O'),
 ('internet', 'NC', 'O'),
 ('que', 'PR', 'O'),
 ('imposibilita', 'VMI', 'O'),
 ('el', 'DA', 'O'),
 ('cumplimiento', 'NC', 'O'),
 ('de', 'SP', 'O'),
 ('los', 'DA', 'O'),
 ('principios', 'NC', 'O'),
 ('básicos', 'AQ', 'O'),
 ('de', 'SP', 'O'),
 ('la', 'DA', 'O'),
 ('Ley', 'NC', 'B-MISC'),
 ('.', 'Fp', 'O')]
[('Las', 'DA', 'O'),
 ('reservas', '

Observe que as palavras estão anotadas seguindo a codificação IOB (Inside, Begin e Outside), ou seja, I para uma palavra "dentro" de uma entidade, B para uma palavra que "começa" uma entidade e O para palavras que estão fora de nomes de entidades, ou seja, que não se tratam de entidades.

Agora vamos definir a função que extrai os nossos atributos de uma dada palavra. A função word2features extrai os atributos mais comuns para tarefas de NER:

* Flag indicando se a primeira letra é maiúscula
* Flag indicando se todas as letras são maiúsculas
* POS Tag da palavra, e a palavra em si (no caso, se transformará em diversos atributos, um para cada POS tag presente e um para cada ocorrência de palavra diferente)
* Os mesmos atributos para a palavra anterior, e para a próxima


In [4]:
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
        'hasnumbers': any(char.isdigit() for char in word)
    }
    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
            '-1:hasnumbers': any(char.isdigit() for char in word1)
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
            '+1:hasnumbers': any(char.isdigit() for char in word1)
        })
    else:
        features['EOS'] = True

    return features


Agora vamos escrever o código para extrair esses atributos do corpus

In [5]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, postag, label in sent]

def sent2tokens(sent):
    return [token for token, postag, label in sent]

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

Vamos ver um exemplo de atributos extraídos

In [6]:
X_train[0][1]

{'bias': 1.0,
 'word.lower()': '(',
 'word[-3:]': '(',
 'word.isupper()': False,
 'word.istitle()': False,
 'word.isdigit()': False,
 'postag': 'Fpa',
 'postag[:2]': 'Fp',
 'hasnumbers': False,
 '-1:word.lower()': 'melbourne',
 '-1:word.istitle()': True,
 '-1:word.isupper()': False,
 '-1:postag': 'NP',
 '-1:postag[:2]': 'NP',
 '-1:hasnumbers': False,
 '+1:word.lower()': 'australia',
 '+1:word.istitle()': True,
 '+1:word.isupper()': False,
 '+1:postag': 'NP',
 '+1:postag[:2]': 'NP',
 '+1:hasnumbers': False}

Agora vamos treinar nosso CRF, usando o algoritmo L-BFGS para descida do gradiente.

In [7]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=20,
    all_possible_states=False # Replica os atributos para cada categoria. 
                              # Isso permite aumentar a performance para categorias específicas, mas deixa o treino mais lento. 
                              # Não vamos usar neste exemplo.
)
crf.fit(X_train, y_train);


Agora vamos avaliar os resultados. Lembrando que a categoria O (não entidades) não é relevante para nós.

In [8]:
from sklearn_crfsuite import metrics

labels = list(crf.classes_)
labels.remove('O')
labels

y_pred = crf.predict(X_test)
metrics.flat_f1_score(y_test, y_pred,
                      average='weighted', labels=labels)

0.612204178501461

In [9]:
# Agrupandos resultados de B e I
sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)
print(metrics.flat_classification_report(
    y_test, y_pred, labels=sorted_labels, digits=3
))

              precision    recall  f1-score   support

       B-LOC      0.686     0.593     0.636      1084
       I-LOC      0.519     0.209     0.298       325
      B-MISC      0.373     0.083     0.135       339
      I-MISC      0.555     0.181     0.273       557
       B-ORG      0.641     0.756     0.694      1400
       I-ORG      0.598     0.796     0.683      1104
       B-PER      0.720     0.769     0.743       735
       I-PER      0.793     0.869     0.829       634

   micro avg      0.657     0.630     0.643      6178
   macro avg      0.611     0.532     0.537      6178
weighted avg      0.637     0.630     0.612      6178



In [10]:
eli5.show_weights(crf, top=30)

<p><b>Exercício 6:</b>Crie um modelo de NER para identificar nomes de jogadores de futebol, no corpus de eventos de narração de jogos.</p>

In [ ]:
import pickle
import os

path = "../input"

x_train = pickle.load(open(os.path.join(path, 'football_ner_x_train.pkl'), 'rb'))
y_train = pickle.load(open(os.path.join(path, 'football_ner_y_train.pkl'), 'rb'))
        
x_test = pickle.load(open(os.path.join(path, 'football_ner_x_test.pkl'), 'rb'))
y_test = pickle.load(open(os.path.join(path, 'football_ner_y_test.pkl'), 'rb'))

In [ ]:
print(x_train[:10])
print(y_train[:10])